Cleaning is not deleting everything unusual. It is translating domain rules into explicit, reviewable transformations. A reliable cleaning notebook preserves raw data, measures each problem, applies a justified rule, and checks the result.

## Learning goals

You will profile missingness, normalize text, parse mixed types, remove exact duplicates, distinguish impossible values from rare values, impute with group context, and write validation checks.

In [1]:
import numpy as np
import pandas as pd

raw = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C03", "C04", "C05", "C06", "C07"],
    "city": [" Bengaluru ", "MUMBAI", "Bengaluru", "Bengaluru", None, "mumbai", "Delhi", "delhi "],
    "age": ["29", "41", "unknown", "unknown", "-4", "37", "212", "33"],
    "monthly_spend": [3200, 5100, np.nan, np.nan, 2800, 4900, 6400, 4500],
    "signup_date": ["2026-01-05", "05/02/2026", "2026-03-12", "2026-03-12",
                    "not recorded", "2026-04-19", "2026-02-28", "2026-06-01"],
})
raw

,customer_id,city,age,monthly_spend,signup_date
0,C01,Bengaluru,29,3200.0,2026-01-05
1,C02,MUMBAI,41,5100.0,05/02/2026
2,C03,Bengaluru,unknown,NaN,2026-03-12
3,C03,Bengaluru,unknown,NaN,2026-03-12
4,C04,NaN,-4,2800.0,not recorded
5,C05,mumbai,37,4900.0,2026-04-19
6,C06,Delhi,212,6400.0,2026-02-28
7,C07,delhi,33,4500.0,2026-06-01


## 1. Profile problems before fixing them

A missing-value count alone is not enough. Inspect types, duplicate rows, value ranges, and suspicious category spellings. Save these measurements so you can later show what changed.

In [2]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
})
print(f"exact duplicate rows: {raw.duplicated().sum()}")
profile

exact duplicate rows: 1


,dtype,missing,unique
customer_id,str,0,7
city,str,1,6
age,str,0,7
monthly_spend,float64,2,6
signup_date,str,0,7


## 2. Normalize representation before comparing values

Whitespace and capitalization create false categories. Normalize them first; map aliases only when you have a controlled vocabulary.

In [3]:
clean = raw.copy()
clean["city"] = clean["city"].astype("string").str.strip().str.title()
clean["city"].value_counts(dropna=False)

city
Bengaluru    3
Mumbai       2
Delhi        2
<NA>         1
Name: count, dtype: int64[pyarrow]

## 3. Parse types and make failures visible

`errors='coerce'` turns invalid representations into missing values. That is useful only if you count the new failures rather than silently accepting them. Mixed date formats should be parsed deliberately.

In [4]:
clean["age"] = pd.to_numeric(clean["age"], errors="coerce")
clean["signup_date"] = pd.to_datetime(clean["signup_date"], format="mixed", errors="coerce")
clean[["age", "signup_date"]].isna().sum().rename("missing_after_parsing")

age            2
signup_date    1
Name: missing_after_parsing, dtype: int64

## 4. Define what duplicate means

Two identical rows are easy. Repeated customer IDs are a domain question: they might be duplicated exports or legitimate repeated events. Here every row is meant to represent one customer snapshot, so an exact repeated row can be removed.

In [5]:
before = len(clean)
clean = clean.drop_duplicates().copy()
print(f"removed {before - len(clean)} exact duplicate row")
print(f"customer IDs still duplicated: {clean.customer_id.duplicated().sum()}")

removed 1 exact duplicate row
customer IDs still duplicated: 0


## 5. Separate impossible from unusual

An age of 212 is not just an outlier for this customer dataset; it violates a defined range. Replace impossible values with missing so the same imputation policy handles them. Do not automatically delete a rare but valid high spender.

In [6]:
valid_age = clean["age"].between(18, 100)
clean.loc[~valid_age & clean["age"].notna(), "age"] = np.nan
clean[["customer_id", "age", "monthly_spend"]]

,customer_id,age,monthly_spend
0,C01,29.0,3200.0
1,C02,41.0,5100.0
2,C03,NaN,NaN
4,C04,NaN,2800.0
5,C05,37.0,4900.0
6,C06,NaN,6400.0
7,C07,33.0,4500.0


## 6. Impute with context—and keep a flag

Imputation invents a value. Preserve that fact in an indicator column. We use the city median when available, then fall back to the overall median.

In [7]:
clean["spend_was_missing"] = clean["monthly_spend"].isna()
city_medians = clean.groupby("city")["monthly_spend"].transform("median")
clean["monthly_spend"] = (
    clean["monthly_spend"]
    .fillna(city_medians)
    .fillna(clean["monthly_spend"].median())
)
clean["age"] = clean["age"].fillna(clean["age"].median())
clean

,customer_id,city,age,monthly_spend,signup_date,spend_was_missing
0,C01,Bengaluru,29.0,3200.0,2026-01-05,False
1,C02,Mumbai,41.0,5100.0,2026-05-02,False
2,C03,Bengaluru,35.0,3200.0,2026-03-12,True
4,C04,<NA>,35.0,2800.0,NaT,False
5,C05,Mumbai,37.0,4900.0,2026-04-19,False
6,C06,Delhi,35.0,6400.0,2026-02-28,False
7,C07,Delhi,33.0,4500.0,2026-06-01,False


## 7. Turn expectations into checks

A pipeline is trustworthy when invalid output causes a loud failure. These assertions are a lightweight data contract.

In [8]:
assert clean["customer_id"].is_unique
assert clean["age"].between(18, 100).all()
assert clean["monthly_spend"].ge(0).all()
assert clean["city"].dropna().isin(["Bengaluru", "Mumbai", "Delhi"]).all()

quality_report = pd.Series({
    "rows_before": len(raw),
    "rows_after": len(clean),
    "remaining_missing_cells": int(clean.isna().sum().sum()),
    "imputed_spend_values": int(clean.spend_was_missing.sum()),
})
quality_report

rows_before                8
rows_after                 7
remaining_missing_cells    2
imputed_spend_values       1
dtype: int64

## Cleaning checklist

- Preserve the raw input.
- Profile before and after.
- Normalize representation before counting categories.
- Record parsing failures.
- Define duplicate keys from the meaning of one row.
- Distinguish invalid values from uncommon values.
- Flag imputed observations.
- Fail loudly when the output violates a contract.

**Try it yourself:** add an invalid city spelling and a negative spend value. Extend the pipeline so both are detected, then write a quality-report row for each rule. Continue to **[Exploratory data analysis](exploratory-analysis.html)**.